In [ ]:
import pandas as pd
df = pd.read_csv('/home/wagyu0923/project/Document_Analyzer/evaluation_data.csv')


/home/wagyu0923/miniconda3/envs/exaone/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import sys
sys.path.append('/home/wagyu0923/project/Document_Analyzer')
import main

file_path = '/home/wagyu0923/project/Document_Analyzer/pdf_files/[세토피아][정정]반기보고서(2025.09.09).pdf'
chunker, embedder, retriever, generator = main.setup_pipeline()
main.run_indexing(file_path, chunker, embedder, retriever)


Chunking Complete
Embedding Complete
Retrieving Coplete


In [ ]:
import json
dataset = df.copy()
for index, query in enumerate(df['user_input']):
    retrieved_data = retriever.retrieve(query, n_results = 5)
    outputs = generator.generate(retrieved_data, query)
    try:
        outputs = json.loads(outputs)
    except json.JSONDecodeError:
        print(f'JSON Decode Error at index {index+1}. Skipping.') 
        print(outputs)
        continue 
    
    if 'answer' not in outputs.keys():
        outputs['answer'] = ''
    dataset.loc[index, 'retrieved_contexts'] = retrieved_data
    dataset.loc[index, 'response'] = outputs['answer']
    print(f'progress : {index+1}/{len(df)}')
    print(outputs)


In [15]:
import ast  
import json 
import pandas as pd 

dataset["retrieved_contexts"] = dataset["retrieved_contexts"].apply(
    lambda x: []
    if x is None or (isinstance(x, float) and pd.isna(x)) or x == ""
    else (ast.literal_eval(x.strip()) if isinstance(x, str) and x.strip().startswith("[") and x.strip().endswith("]")
          else (x if isinstance(x, list) else [x]))
)

if "reference" in dataset.columns:
    dataset["reference"] = dataset["reference"].apply(
        lambda x: "" if x is None or (isinstance(x, float) and pd.isna(x))
        else (x if isinstance(x, str) else "\n\n".join(map(str, x)))
    )
    
dataset.fillna('',inplace=True)

In [ ]:
import re

def strip_source_block(text: str) -> str:
    if not isinstance(text, str):
        return text
    
    blocks = text.split("source :")
    contents = []

    for block in blocks:
        block = block.strip()
        if not block:
            continue

        if "content :" in block:
            content_part = block.split("content :", 1)[1].strip()
            contents.append(content_part)
        else:
        
            contents.append(block)


    return "\n\n".join(contents)


def clean_retrieved_contexts(x):
   
    if isinstance(x, list):
        return [strip_source_block(t) for t in x]
    
    elif isinstance(x, str):
        return [strip_source_block(x)]

    else:
        return []



dataset["retrieved_contexts"] = dataset["retrieved_contexts"].apply(clean_retrieved_contexts)

In [21]:
dataset.to_csv('response_data.csv')

In [23]:
import os
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

from ragas import EvaluationDataset, evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
)
from ragas.run_config import RunConfig


evaluation_dataset = EvaluationDataset.from_pandas(dataset)


from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper

base_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key = OPENAI_API_KEY        
)

evaluator_llm = LangchainLLMWrapper(base_llm)


metrics = [
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
]

result = evaluate(
    dataset=evaluation_dataset,
    metrics=metrics,
    llm=evaluator_llm,         
)

print(result)


/tmp/ipykernel_29737/842449243.py:28: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(base_llm)
Evaluating:  22%|██▏       | 26/120 [00:58<02:37,  1.67s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  33%|███▎      | 40/120 [01:19<01:46,  1.33s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  50%|█████     | 60/120 [01:54<01:27,  1.45s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generatio

{'context_precision': 0.8333, 'context_recall': 0.7583, 'faithfulness': 0.7816, 'answer_relevancy': 0.4467}


In [10]:
import pandas as pd
import sys
sys.path.append('/home/wagyu0923/project/Document_Analyzer')


result_df = result.to_pandas()
result_df.to_csv('result.csv')
result_df

NameError: name 'result' is not defined

In [23]:
import pandas as pd
import sys
result_bm25 = pd.read_csv('/home/wagyu0923/project/Document_Analyzer/result_bm25.csv')
result_bm25_mean = result_bm25[['context_precision','context_recall','faithfulness','answer_relevancy']].mean()
print(result_bm25_mean)

context_precision    0.966667
context_recall       0.783333
faithfulness         0.705556
answer_relevancy     0.521057
dtype: float64


In [24]:
result_vector = pd.read_csv('/home/wagyu0923/project/Document_Analyzer/result_vector.csv')
result_vector_mean = result_vector[['context_precision','context_recall','faithfulness','answer_relevancy']].mean()
print(result_vector_mean)


context_precision    0.900000
context_recall       0.783333
faithfulness         0.724138
answer_relevancy     0.447672
dtype: float64
